In [1]:
# General
import os
import numpy as np
import matplotlib.pyplot as plt
import random
# model
import tensorflow
import tensorflow as tf
import pickle
import time
import sys

import keras
from keras.models import Sequential
from keras.layers import Add, GlobalAveragePooling2D,Dense, Flatten, Conv2D, Lambda,Input, BatchNormalization, Activation
from tensorflow.keras.utils import Sequence
from keras.utils import to_categorical
# from keras.callbacks import TensorBoard
from tensorflow.keras.layers import DepthwiseConv2D

from tensorflow.keras.optimizers import AdamW
from tensorflow.keras import *
# from tensorflow.keras import layers as Layers
from tensorflow.keras.layers import *
from keras.activations import relu
from tensorflow.keras.layers import *
from tensorflow.keras import layers, models

# from tensorflow.keras.layers import Add, GlobalAveragePooling2D,Dense, Flatten, Conv2D, Lambda,	Input, BatchNormalization, Activation
from tensorflow.keras.preprocessing.image import ImageDataGenerator
# from keras.utils import to_categorical
from tensorflow.keras.optimizers import *
from tensorflow.keras.callbacks import *
from tensorflow.keras.callbacks import TensorBoard, ModelCheckpoint,EarlyStopping,ReduceLROnPlateau,LambdaCallback

2024-06-04 10:27:19.174555: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2024-06-04 10:27:19.174695: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2024-06-04 10:27:19.301539: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


## Config

In [2]:
policy = mixed_precision.Policy('mixed_float16')
mixed_precision.set_global_policy(policy)

In [3]:
sys.path.append('/kaggle/input/resnet-34/')
directory = "/kaggle/working/"
model_dir = "/kaggle/working/"
weight_decay = 1e-4
lr=[5e-3,1e-3,5e-4,1e-4]
num_classes = 100
dropout_rate=0.

BATCH_SIZE = 32
IMG_SIZE = 32
optimizer = AdamW
epoch = 50
loss='categorical_crossentropy'
lr_reducer = ReduceLROnPlateau(factor=np.sqrt(0.1),
                               cooldown=0,
                               patience=5,
                               min_lr=0.5e-6)

# early_stopping = EarlyStopping(monitor='val_accuracy', min_delta=0.001, patience=6, verbose=0, mode='auto', baseline=None, restore_best_weights=True)
# early_stopping = EarlyStopping(monitor='val_loss', min_delta=0.001, patience=5, verbose=0, mode='auto', baseline=None, restore_best_weights=True)

# Data + Normal Augmentation

In [4]:
BATCH_SIZE = 32
IMG_SIZE =32
(X_train, y_train), (X_test, y_test) = cifar100.load_data()
# Convert class vectors to binary class matrices.
y_train = to_categorical(y_train, num_classes=num_classes)
y_test = to_categorical(y_test, num_classes=num_classes)


X_train = X_train.astype('float32')
X_test = X_test.astype('float32')


X_train /= 255.
X_test /= 255.


steps = int(X_train.shape[0] / BATCH_SIZE)
steps_per_epoch = len(X_train) // BATCH_SIZE


datagen = ImageDataGenerator(
    rotation_range=15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    horizontal_flip=True,
)
class CutMixImageDataGenerator():
    def __init__(self, generator1, generator2, img_size, batch_size):
        self.batch_index = 0
        self.samples = generator1.n
        self.generator1 = generator1
        self.generator2 = generator2
        self.img_size = img_size
        self.batch_size = batch_size

    def reset_index(self):
        self.generator1._set_index_array()
        self.generator2._set_index_array()

    def reset(self):
        self.batch_index = 0
        self.generator1.reset()
        self.generator2.reset()
        self.reset_index()

    def get_steps_per_epoch(self):
        quotient, remainder = divmod(self.samples, self.batch_size)
        return (quotient + 1) if remainder else quotient

    def __len__(self):
        return self.get_steps_per_epoch()

    def __next__(self):
        if self.batch_index == 0:
            self.reset()

        crt_idx = self.batch_index * self.batch_size
        if crt_idx >= self.samples:
            self.batch_index = 0
            raise StopIteration

        reshape_size = self.batch_size
        last_step_start_idx = (self.get_steps_per_epoch() - 1) * self.batch_size
        if crt_idx == last_step_start_idx:
            reshape_size = self.samples - last_step_start_idx

        X_1, y_1 = self.generator1.__next__()
        X_2, y_2 = self.generator2.__next__()

        cut_ratio = np.random.beta(a=1, b=1, size=reshape_size)
        cut_ratio = np.clip(cut_ratio, 0.3, 0.7)

        label_ratio = cut_ratio.reshape(reshape_size, 1)
        cut_img = X_2

        X = X_1.copy()
        for i in range(reshape_size):
            cut_size = int((self.img_size - 1) * cut_ratio[i])
            y1 = tf.random.uniform(shape=[], minval=0, maxval=(self.img_size - 1) - cut_size, dtype=tf.int32).numpy()
            x1 = tf.random.uniform(shape=[], minval=0, maxval=(self.img_size - 1) - cut_size, dtype=tf.int32).numpy()
            y2 = y1 + cut_size
            x2 = x1 + cut_size
            cut_arr = cut_img[i][y1:y2, x1:x2]
            X[i][y1:y2, x1:x2] = cut_arr

        y = y_1 * (1 - (label_ratio ** 2)) + y_2 * (label_ratio ** 2)   #### smoothing the labels
        self.batch_index += 1
        return X, y

    def __iter__(self):
        return self

train_generator1 = datagen.flow(X_train, y_train, batch_size=BATCH_SIZE, shuffle=True)  ####create augmented images twice from the normal augments
train_generator2 = datagen.flow(X_train, y_train, batch_size=BATCH_SIZE, shuffle=True)

train_generator = CutMixImageDataGenerator(
    generator1=train_generator1,
    generator2=train_generator2,
    img_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
)

def generator_wrapper(generator):
    first_run = True
    while True:
        try:
            X, y = next(generator)
            if first_run:
                print(f'X shape: {X.shape}, y shape: {y.shape}')
                first_run = False  # Set the flag to False after printing
            if X.shape[0] == BATCH_SIZE:
                yield X, y
            elif X.shape[0] < BATCH_SIZE:
                # Pad the batch if it is smaller than BATCH_SIZE
                X = np.pad(X, ((0, BATCH_SIZE - X.shape[0]), (0, 0), (0, 0), (0, 0)), 'constant')
                y = np.pad(y, ((0, BATCH_SIZE - y.shape[0]), (0, 0)), 'constant')
                yield X, y
        except StopIteration:
            generator.reset()

train_ds = tf.data.Dataset.from_generator(
    lambda: generator_wrapper(train_generator),
    output_signature=(
        tf.TensorSpec(shape=(BATCH_SIZE, IMG_SIZE, IMG_SIZE, 3), dtype=tf.float32),
        tf.TensorSpec(shape=(BATCH_SIZE, num_classes), dtype=tf.float32),
    )
)

train_ds = (train_ds
            .shuffle(1024)
            .cache()
            .repeat()
            .prefetch(buffer_size=tf.data.experimental.AUTOTUNE))

170498071/170498071 ━━━━━━━━━━━━━━━━━━━━ 8s 0us/step


# Model ConvMix

In [5]:
import tensorflow as tf
from tensorflow.keras import layers, models

def activation_block(x, dropout_rate, training):
    x = layers.Activation("gelu")(x)
    x = layers.Dropout(dropout_rate)(x, training=training)
    return layers.BatchNormalization()(x, training=training)

def conv_stem(x, filters: int, patch_size: int, dropout_rate, training):
    x = layers.Conv2D(filters, kernel_size=patch_size, strides=patch_size)(x)
    return activation_block(x, dropout_rate, training)

def conv_mixer_block(x, filters: int, kernel_size: int, dropout_rate, training):
    # Depthwise convolution.
    x0 = x
    x = layers.DepthwiseConv2D(kernel_size=kernel_size, padding="same")(x)
    x = layers.Dropout(dropout_rate)(x, training=training)
    x = layers.Add()([activation_block(x, dropout_rate, training), x0])  # Residual.

    # Pointwise convolution.
    x = layers.Conv2D(filters, kernel_size=1)(x)
    x = layers.Dropout(dropout_rate)(x, training=training)
    x = activation_block(x, dropout_rate, training)

    return x

def get_conv_mixer_512_12(
    image_size=32, filters=512, depth=12, kernel_size=3, patch_size=2, num_classes=num_classes, dropout_rate=dropout_rate
):
    """ConvMixer-256/8: https://openreview.net/pdf?id=TVHS5Y4dNvM.
    The hyperparameter values are taken from the paper.
    """
    inputs = layers.Input((image_size, image_size, 3))
#     x = layers.Rescaling(scale=1.0 / 255)(inputs)

    # Extract patch embeddings.
    x = conv_stem(inputs, filters, patch_size, dropout_rate, training=True)

    # ConvMixer blocks.
    for _ in range(depth):
        x = conv_mixer_block(x, filters, kernel_size, dropout_rate, training=True)

    # Classification block.
    x = layers.GlobalAvgPool2D()(x)
    x = layers.Dropout(dropout_rate)(x, training=True)  # Dropout with rate 0 before final dense layer
    outputs = layers.Dense(num_classes, activation="softmax")(x)

    return models.Model(inputs, outputs)

model = get_conv_mixer_512_12(dropout_rate=dropout_rate)

# model.summary()

## Setting up Hyper params

In [6]:
class GCAdamW(AdamW):
    def __init__(self, learning_rate, weight_decay, **kwargs):
        super(GCAdamW, self).__init__(learning_rate, weight_decay, **kwargs)

    def get_gradients(self, loss, params):
        # Compute the gradients
        grads = super().get_gradients(loss, params)

        # Centralize the gradients
        centralized_grads = []
        for grad in grads:
            if grad is not None:
                grad_len = len(grad.shape)
                if grad_len > 1:
                    axis = list(range(grad_len - 1))
                    grad -= tf.reduce_mean(grad, axis=axis, keepdims=True)
            centralized_grads.append(grad)
        
        return centralized_grads
# optimizer = GCAdamW(learning_rate=0.001, weight_decay=1e-4)

In [7]:

for i in range(len(lr)):
    filepath=f"Bv{lr[i]}.weights.h5"
checkpoint = ModelCheckpoint(filepath=filepath,
                             monitor='val_acc',
                             verbose=1,
                             save_best_only=True,
                            save_weights_only= True)


lr_reducer = ReduceLROnPlateau(factor=np.sqrt(0.1),
                               cooldown=0,
                               patience=5,
                               min_lr=0.5e-6)

callbacks = [checkpoint  ]

### training

In [8]:

model = get_conv_mixer_512_12( dropout_rate=dropout_rate)


filename = f"ConvMix_10_CutMix_{lr[0]:.0e}.weights.h5"


filepath = os.path.join(directory, filename)
checkpoint = ModelCheckpoint(filepath=filepath,
                                      monitor='val_accuracy',
                                      verbose=1,
                                      save_best_only=True,
                                      save_weights_only=True)



optimizer = GCAdamW(learning_rate=lr[0],weight_decay=weight_decay)


callbacks = [checkpoint, ]


model.compile(
    optimizer=optimizer,
    loss=keras.losses.CategoricalCrossentropy(),
    metrics=['accuracy'],
    )


history =model.fit(train_ds , steps_per_epoch=1360 , epochs=50, validation_data=(X_test, y_test), callbacks=callbacks)


history_save_path = os.path.join(model_dir, f'ConvMixer_10_CutMix_{lr[0]:.0e}.pkl')
with open(history_save_path, 'wb') as file_pi:
    pickle.dump(history.history, file_pi)

Epoch 1/50
X shape: (32, 32, 32, 3), y shape: (32, 10)
   2/1360 ━━━━━━━━━━━━━━━━━━━━ 1:17 57ms/step - accuracy: 0.1094 - loss: 2.5247   

I0000 00:00:1717496951.334542      99 device_compiler.h:186] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


1360/1360 ━━━━━━━━━━━━━━━━━━━━ 0s 66ms/step - accuracy: 0.2548 - loss: 2.1304
Epoch 1: val_accuracy improved from -inf to 0.35380, saving model to /kaggle/working/ConvMix_10_CutMix_5e-03.weights.h5
1360/1360 ━━━━━━━━━━━━━━━━━━━━ 187s 72ms/step - accuracy: 0.2548 - loss: 2.1303 - val_accuracy: 0.3538 - val_loss: 2.1026
Epoch 2/50
1360/1360 ━━━━━━━━━━━━━━━━━━━━ 0s 66ms/step - accuracy: 0.3716 - loss: 1.8583
Epoch 2: val_accuracy did not improve from 0.35380
1360/1360 ━━━━━━━━━━━━━━━━━━━━ 93s 69ms/step - accuracy: 0.3716 - loss: 1.8583 - val_accuracy: 0.3455 - val_loss: 2.0439
Epoch 3/50
1360/1360 ━━━━━━━━━━━━━━━━━━━━ 0s 66ms/step - accuracy: 0.4135 - loss: 1.7595
Epoch 3: val_accuracy improved from 0.35380 to 0.49840, saving model to /kaggle/working/ConvMix_10_CutMix_5e-03.weights.h5
1360/1360 ━━━━━━━━━━━━━━━━━━━━ 93s 69ms/step - accuracy: 0.4135 - loss: 1.7594 - val_accuracy: 0.4984 - val_loss: 1.5254
Epoch 4/50
1360/1360 ━━━━━━━━━━━━━━━━━━━━ 0s 66ms/step - accuracy: 0.4535 - loss: 1.67

In [9]:

# model = get_conv_mixer_512_12( dropout_rate=dropout_rate)


# filename = f"ConvMix_10_CutMix_{lr[1]:.0e}.weights.h5"


# filepath = os.path.join(directory, filename)
# checkpoint = ModelCheckpoint(filepath=filepath,
#                                       monitor='val_accuracy',
#                                       verbose=1,
#                                       save_best_only=True,
#                                       save_weights_only=True)



# optimizer = GCAdamW(learning_rate=lr[1],weight_decay=weight_decay)


# callbacks = [checkpoint, ]


# model.compile(
#     optimizer=optimizer,
#     loss=keras.losses.CategoricalCrossentropy(),
#     metrics=['accuracy'],
#     )


# history =model.fit(train_ds , steps_per_epoch=1360 , epochs=50, validation_data=(X_test, y_test), callbacks=callbacks)


# history_save_path = os.path.join(model_dir, f'ConvMixer_10_CutMix_{lr[1]:.0e}.pkl')
# with open(history_save_path, 'wb') as file_pi:
#     pickle.dump(history.history, file_pi)

In [10]:

# model = get_conv_mixer_512_12( dropout_rate=dropout_rate)


# filename = f"ConvMix_10_CutMix_{lr[2]:.0e}.weights.h5"


# filepath = os.path.join(directory, filename)
# checkpoint = ModelCheckpoint(filepath=filepath,
#                                       monitor='val_accuracy',
#                                       verbose=1,
#                                       save_best_only=True,
#                                       save_weights_only=True)



# optimizer = GCAdamW(learning_rate=lr[2],weight_decay=weight_decay)


# callbacks = [checkpoint, ]


# model.compile(
#     optimizer=optimizer,
#     loss=keras.losses.CategoricalCrossentropy(),
#     metrics=['accuracy'],
#     )


# history =model.fit(train_ds , steps_per_epoch=1360 , epochs=50, validation_data=(X_test, y_test), callbacks=callbacks)


# history_save_path = os.path.join(model_dir, f'ConvMixer_10_CutMix_{lr[2]:.0e}.pkl')
# with open(history_save_path, 'wb') as file_pi:
#     pickle.dump(history.history, file_pi)

In [11]:

# model = get_conv_mixer_512_12( dropout_rate=dropout_rate)


# filename = f"ConvMix_10_CutMix_{lr[3]:.0e}.weights.h5"


# filepath = os.path.join(directory, filename)
# checkpoint = ModelCheckpoint(filepath=filepath,
#                                       monitor='val_accuracy',
#                                       verbose=1,
#                                       save_best_only=True,
#                                       save_weights_only=True)



# optimizer = GCAdamW(learning_rate=lr[3],weight_decay=weight_decay)


# callbacks = [checkpoint, ]


# model.compile(
#     optimizer=optimizer,
#     loss=keras.losses.CategoricalCrossentropy(),
#     metrics=['accuracy'],
#     )


# history =model.fit(train_ds , steps_per_epoch=1360 , epochs=50, validation_data=(X_test, y_test), callbacks=callbacks)


# history_save_path = os.path.join(model_dir, f'ConvMixer_10_CutMix_{lr[3]:.0e}.pkl')
# with open(history_save_path, 'wb') as file_pi:
#     pickle.dump(history.history, file_pi)